# Abaya & Thobe Recommendation System
### Hybrid ML Recommender — SVD Collaborative Filter + TF-IDF Content Model + MobileNetV2 Visual Search

Just run every cell top-to-bottom.

| # | Cell | What it does |
|---|------|-------------|
| 1 | Setup | Mount Drive, create folder structure |
| 2 | Dependencies | Install all packages |
| 3 | Product Catalog | Generate 100 synthetic Abayas & Thobes |
| 4 | User Interactions | Generate 200 users × behaviour events |
| 5 | Images | Download & preprocess real garment images (Bing) |
| 6 | SVD Model | Train collaborative filter |
| 7 | TF-IDF Model | Train content-based similarity model |
| 8 | CNN Model | Fine-tune MobileNetV2 + extract visual features |
| 9 | Write API | Write FastAPI server to disk |
| 10 | Start Server | Launch uvicorn + cloudflared public tunnel |
| 11 | Test Endpoints | Verify all 6 API routes |


## Mount Drive & Create Project Structure

In [ ]:
from google.colab import drive
import os, sys

drive.mount('/content/drive')

BASE = '/content/drive/MyDrive/thobe_recommender'

# create folder
dirs = [
    'data/raw/images/abaya',
    'data/raw/images/thobe',
    'data/raw/metadata',
    'data/processed/images',
    'models/cv_model',
    'models/recommender',
    'src/api',
]
for d in dirs:
    os.makedirs(f'{BASE}/{d}', exist_ok=True)

# init files
for pkg in ['src', 'src/api']:
    init = f'{BASE}/{pkg}/__init__.py'
    if not os.path.exists(init):
        open(init, 'w').close()

os.chdir(BASE)
sys.path.insert(0, BASE)

print(f'Drive mounted')
print(f' Project root : {BASE}')
print(f' All folders created')
print()


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive mounted
 Project root : /content/drive/MyDrive/thobe_recommender
 All folders created



## Install Dependencies

In [ ]:
import subprocess, sys

def pip(*pkgs):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *pkgs], check=True)

print('Installing packages ...')
pip('fastapi==0.104.1', 'uvicorn==0.24.0',
    'python-multipart==0.0.6', 'pydantic==2.5.0')
pip('icrawler==0.6.6')
print(' Packages installed')

import numpy as np, torch, sklearn
print(f' NumPy    {np.__version__}')
print(f' PyTorch  {torch.__version__}')
print(f' sklearn  {sklearn.__version__}')
print()


Installing packages ...
 Packages installed
 NumPy    2.0.2
 PyTorch  2.10.0+cu128
 sklearn  1.6.1



## Generate Synthetic Product Catalog (100 items)

In [ ]:
import pandas as pd
import numpy as np
import os

BASE = '/content/drive/MyDrive/thobe_recommender'

if os.path.exists(f'{BASE}/data/raw/metadata/products.csv'):
    print(' Products already exist — skipping generation')
    df = pd.read_csv(f'{BASE}/data/raw/metadata/products.csv')
    print(f'Loaded {len(df)} existing products')
else:
    np.random.seed(42)

    garment_types  = ['abaya', 'thobe']
    occasions      = ['casual', 'formal', 'wedding', 'religious', 'everyday']
    fabrics        = ['cotton', 'linen', 'silk', 'polyester', 'crepe', 'chiffon']
    colors_abaya   = ['black', 'navy', 'grey', 'dark_brown', 'charcoal']
    colors_thobe   = ['white', 'cream', 'beige', 'grey', 'off_white']
    patterns       = ['plain', 'embroidered', 'printed', 'striped', 'geometric']
    sleeves        = ['long', 'short', '3/4', 'sleeveless']
    embellishments = ['none', 'lace', 'embroidery', 'buttons', 'beading', 'sequins']
    price_ranges   = ['budget', 'mid_range', 'premium', 'luxury']
    brands         = ['AlNoor', 'Elegance', 'Royal Abaya', 'Classic Thobe',
                      'Modesty', 'Gulf Style', 'AlFarasha']
    price_map      = {
        'budget':    (80,  200),
        'mid_range': (200, 400),
        'premium':   (400, 700),
        'luxury':    (700, 1200),
    }

    products = []
    for i in range(100):
        gtype  = garment_types[i % 2]
        colors = colors_abaya if gtype == 'abaya' else colors_thobe
        pr     = np.random.choice(price_ranges)
        price  = round(float(np.random.uniform(*price_map[pr])), 2)
        color  = np.random.choice(colors)
        pat    = np.random.choice(patterns)
        products.append({
            'product_id':    f'p{str(i).zfill(3)}',
            'garment_type':  gtype,
            'occasion':      np.random.choice(occasions),
            'fabric':        np.random.choice(fabrics),
            'color':         color,
            'pattern':       pat,
            'sleeve_length': np.random.choice(sleeves),
            'embellishment': np.random.choice(embellishments),
            'price_range':   pr,
            'brand':         np.random.choice(brands),
            'price':         price,
            'currency':      'SAR',
            'title':         f'{color.title()} {pat.title()} {gtype.title()}',
        })

    df = pd.DataFrame(products)
    os.makedirs(f'{BASE}/data/raw/metadata', exist_ok=True)
    df.to_csv(f'{BASE}/data/raw/metadata/products.csv', index=False)

    print(f' Generated {len(df)} products')
    print(f"  Abayas : {len(df[df.garment_type=='abaya'])}")
    print(f"  Thobes : {len(df[df.garment_type=='thobe'])}")

print()
print(df[['product_id','garment_type','color','occasion','price']].head(6).to_string())
print()

 Products already exist — skipping generation
Loaded 100 existing products

  product_id garment_type  color   occasion    price
0       p000        abaya   grey   everyday   638.96
1       p001        thobe  beige     formal   866.85
2       p002        abaya  black     formal  1196.11
3       p003        thobe  cream  religious   142.97
4       p004        abaya  black   everyday   957.12
5       p005        thobe  white     formal   403.98



## Generate Synthetic User Interactions

In [ ]:
import pandas as pd
import numpy as np
import os

BASE = '/content/drive/MyDrive/thobe_recommender'

if os.path.exists(f'{BASE}/data/raw/metadata/interactions.parquet'):
    print(' Interactions already exist — skipping generation')
    df_int = pd.read_parquet(f'{BASE}/data/raw/metadata/interactions.parquet')
    print(f'  Loaded {len(df_int):,} existing interactions')
else:
    df_prods = pd.read_csv(f'{BASE}/data/raw/metadata/products.csv')
    np.random.seed(99)

    segments = {
        'wedding_shopper': {'occasions': ['wedding','formal'],
                            'price':     ['premium','luxury'],   'count': 50},
        'daily_wear':      {'occasions': ['everyday','casual'],
                            'price':     ['budget','mid_range'], 'count': 80},
        'religious_buyer': {'occasions': ['religious','formal'],
                            'price':     ['mid_range','premium'],'count': 40},
        'gift_buyer':      {'occasions': ['wedding','everyday'],
                            'price':     ['premium'],            'count': 30},
    }

    interactions = []
    uid = 0
    for segment, props in segments.items():
        seg_prods = df_prods[
            df_prods['occasion'].isin(props['occasions']) &
            df_prods['price_range'].isin(props['price'])
        ]['product_id'].tolist()
        if not seg_prods:
            seg_prods = df_prods['product_id'].tolist()

        for _ in range(props['count']):
            user_id  = f'user_{str(uid).zfill(4)}'
            n_events = np.random.randint(5, 30)
            for _ in range(n_events):
                interactions.append({
                    'user_id':    user_id,
                    'product_id': np.random.choice(seg_prods),
                    'event_type': np.random.choice(
                        ['view','click','add_to_cart','purchase'],
                        p=[0.60, 0.25, 0.10, 0.05]),
                    'score':   np.random.randint(1, 6),
                    'segment': segment,
                })
            uid += 1

    df_int = pd.DataFrame(interactions)
    os.makedirs(f'{BASE}/data/raw/metadata', exist_ok=True)
    df_int.to_parquet(f'{BASE}/data/raw/metadata/interactions.parquet')

    print(f' Generated {len(df_int):,} interaction events')
    print(f' {uid} users across {len(segments)} segments')

print()
print('Event breakdown:')
print(df_int['event_type'].value_counts().to_string())
print()

 Interactions already exist — skipping generation
  Loaded 3,294 existing interactions

Event breakdown:
event_type
view           1953
click           879
add_to_cart     317
purchase        145



## Download & Preprocess Garment Images (Bing)

In [ ]:
import os
from icrawler.builtin import BingImageCrawler
from PIL import Image

BASE = '/content/drive/MyDrive/thobe_recommender'

tasks = [
    ('abaya', ['abaya black women modest dress',
               'abaya embroidered women fashion']),
    ('thobe', ['thobe white men arabic traditional',
               'kandura men gulf dress white']),
]

TARGET_PER_CLASS = 250

for garment, keywords in tasks:
    img_dir  = f'{BASE}/data/raw/images/{garment}'
    existing = len([f for f in os.listdir(img_dir) if f.endswith('.jpg')])
    needed   = max(0, TARGET_PER_CLASS - existing)

    if needed == 0:
        print(f' {garment}: already has {existing} images skipping download')
        continue

    print(f'Downloading {needed} {garment} images ...')
    per_kw = needed // len(keywords) + 10

    downloaded = 0
    for kw in keywords:
        if downloaded >= needed:
            break
        tmp = f'{BASE}/data/raw/images/{garment}_tmp'
        os.makedirs(tmp, exist_ok=True)

        try:
            crawler = BingImageCrawler(
                feeder_threads=1, parser_threads=1, downloader_threads=4,
                storage={'root_dir': tmp})
            crawler.crawl(keyword=kw, max_num=per_kw,
                          min_size=(100,100), filters=dict(type='photo'))
        except Exception as e:
            print(f'crawler error: {e}')

        for fname in os.listdir(tmp):
            if downloaded >= needed:
                break
            src = os.path.join(tmp, fname)
            try:
                img = Image.open(src).convert('RGB').resize((224,224))
                dst = os.path.join(img_dir,
                      f'{garment}_{existing+downloaded:04d}.jpg')
                img.save(dst, 'JPEG', quality=90)
                downloaded += 1
            except:
                pass
            finally:
                try: os.remove(src)
                except: pass
        try: os.rmdir(tmp)
        except: pass

    print(f' Downloaded {downloaded} new {garment} images')

for garment in ['abaya','thobe']:
    n = len([f for f in os.listdir(f'{BASE}/data/raw/images/{garment}')
             if f.endswith('.jpg')])
    print(f'  {garment}: {n} images total')

# copy all raw images to processed for feature extraction
proc_dir = f'{BASE}/data/processed/images'
count = 0
for garment in ['abaya','thobe']:
    src_dir = f'{BASE}/data/raw/images/{garment}'
    for fname in sorted(os.listdir(src_dir)):
        if not fname.endswith('.jpg'):
            continue
        try:
            img = Image.open(os.path.join(src_dir,fname)).convert('RGB')
            img = img.resize((224,224))
            img.save(os.path.join(proc_dir, f'{garment}_{count:04d}.jpg'),
                     'JPEG', quality=90)
            count += 1
        except:
            pass
print(f'{count} images in data/processed/images/')
print()


 abaya: already has 250 images skipping download
 thobe: already has 250 images skipping download
  abaya: 250 images total
  thobe: 250 images total
500 images in data/processed/images/



## Train SVD Collaborative Filter

In [ ]:
import pandas as pd
import numpy as np
import pickle, os

BASE = '/content/drive/MyDrive/thobe_recommender'
np.random.seed(42)

df_int = pd.read_parquet(f'{BASE}/data/raw/metadata/interactions.parquet')
score_map = {'view':1,'click':2,'add_to_cart':4,'purchase':5}
df_int['rating'] = df_int['event_type'].map(score_map)

users    = df_int['user_id'].unique().tolist()
products = df_int['product_id'].unique().tolist()
user2idx = {u:i for i,u in enumerate(users)}
item2idx = {p:i for i,p in enumerate(products)}
n_users, n_items, n_factors = len(users), len(products), 50

print(f' {n_users} users, {n_items} products')

U  = np.random.normal(0, 0.1, (n_users,  n_factors))
V  = np.random.normal(0, 0.1, (n_items,  n_factors))
bu = np.zeros(n_users)
bi = np.zeros(n_items)
mu = df_int['rating'].mean()

lr, reg, epochs = 0.005, 0.02, 20
print(f'Training SVD ({epochs} epochs) ...')

for epoch in range(epochs):
    total_loss, count = 0.0, 0
    for _, row in df_int.iterrows():
        u = user2idx.get(row['user_id'])
        i = item2idx.get(row['product_id'])
        if u is None or i is None:
            continue
        r   = row['rating']
        err = r - (mu + bu[u] + bi[i] + U[u] @ V[i])
        bu[u] += lr * (err - reg * bu[u])
        bi[i] += lr * (err - reg * bi[i])
        U[u]  += lr * (err * V[i] - reg * U[u])
        V[i]  += lr * (err * U[u] - reg * V[i])
        total_loss += err**2; count += 1
    if (epoch+1) % 5 == 0:
        print(f'  Epoch {epoch+1:02d}/{epochs} — RMSE: {np.sqrt(total_loss/count):.4f}')

svd_model = dict(U=U, V=V, bu=bu, bi=bi, mu=mu,
                 user2idx=user2idx, item2idx=item2idx,
                 users=users, products=products)
os.makedirs(f'{BASE}/models/recommender', exist_ok=True)
pickle.dump(svd_model, open(f'{BASE}/models/recommender/svd_model.pkl','wb'))
print('SVD model saved')

# test
print('\nSample predictions:')
for uid, pid in [('user_0001','p000'),('user_0050','p010')]:
    u = user2idx.get(uid); i = item2idx.get(pid)
    if u is not None and i is not None:
        s = float(np.clip(mu+bu[u]+bi[i]+U[u]@V[i], 1, 5))
        print(f'  {uid} - {pid} : {s:.3f}')
print()


 200 users, 57 products
Training SVD (20 epochs) ...
  Epoch 05/20 — RMSE: 1.1053
  Epoch 10/20 — RMSE: 1.0733
  Epoch 15/20 — RMSE: 1.0431
  Epoch 20/20 — RMSE: 1.0104
SVD model saved

Sample predictions:
  user_0001 - p000 : 1.632
  user_0050 - p010 : 1.727



## Train TF-IDF Content-Based Model

In [ ]:
import pandas as pd
import pickle, os
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

BASE = '/content/drive/MyDrive/thobe_recommender'

df_prods = pd.read_csv(f'{BASE}/data/raw/metadata/products.csv')
print(f' {len(df_prods)} products loaded')

df_prods['text'] = (
    df_prods['garment_type']  + ' ' +
    df_prods['occasion']      + ' ' +
    df_prods['fabric']        + ' ' +
    df_prods['color']         + ' ' +
    df_prods['pattern']       + ' ' +
    df_prods['sleeve_length'] + ' ' +
    df_prods['embellishment'] + ' ' +
    df_prods['price_range']   + ' ' +
    df_prods['brand']
)

tfidf     = TfidfVectorizer()
tfidf_mat = tfidf.fit_transform(df_prods['text'])
cos_sim   = cosine_similarity(tfidf_mat, tfidf_mat)
print(f' TF-IDF matrix : {tfidf_mat.shape}')
print(f' Cosine matrix : {cos_sim.shape}')

pickle.dump(tfidf,    open(f'{BASE}/models/recommender/tfidf.pkl',      'wb'))
pickle.dump(cos_sim,  open(f'{BASE}/models/recommender/cosine_sim.pkl', 'wb'))
pickle.dump(df_prods, open(f'{BASE}/models/recommender/products_df.pkl','wb'))
print(' Models saved')

# quick test
prod_ids = df_prods['product_id'].tolist()
idx  = 0
sims = sorted(enumerate(cos_sim[idx]), key=lambda x: x[1], reverse=True)
print(f'\nTop 3 similar to {prod_ids[idx]} ({df_prods["title"][0]}):')
for rank,(i,s) in enumerate(sims[1:4],1):
    print(f'  {rank}. {prod_ids[i]} | {df_prods["title"][i]} | score: {s:.4f}')
print()


 100 products loaded
 TF-IDF matrix : (100, 48)
 Cosine matrix : (100, 100)
 Models saved

Top 3 similar to p000 (Grey Geometric Abaya):
  1. p016 | Grey Geometric Abaya | score: 0.7217
  2. p094 | Grey Striped Abaya | score: 0.6668
  3. p026 | Grey Striped Abaya | score: 0.6036



## Fine-tune MobileNetV2 + Extract Visual Features

In [ ]:
import os, pickle
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader
from PIL import Image

BASE      = '/content/drive/MyDrive/thobe_recommender'
DEVICE    = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
CLASSES   = ['abaya', 'thobe']
print(f' Device: {DEVICE}')

abaya_dir = f'{BASE}/data/raw/images/abaya'
thobe_dir = f'{BASE}/data/raw/images/thobe'
n_abaya   = len([f for f in os.listdir(abaya_dir) if f.endswith('.jpg')])
n_thobe   = len([f for f in os.listdir(thobe_dir) if f.endswith('.jpg')])
print(f' Training images — abaya: {n_abaya}, thobe: {n_thobe}')

# dataset
class GarmentDataset(Dataset):
    def __init__(self, transform):
        self.samples   = []
        self.transform = transform
        for fname in sorted(os.listdir(abaya_dir)):
            if fname.endswith('.jpg'):
                self.samples.append((os.path.join(abaya_dir, fname), 0))
        for fname in sorted(os.listdir(thobe_dir)):
            if fname.endswith('.jpg'):
                self.samples.append((os.path.join(thobe_dir, fname), 1))

    def __len__(self):  return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        try:
            img = Image.open(path).convert('RGB')
        except:
            img = Image.new('RGB', (224,224), (128,128,128))
        return self.transform(img), label

train_tf = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])
eval_tf = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])

# Model
model = models.mobilenet_v2(weights='IMAGENET1K_V1')
for p in model.parameters():
    p.requires_grad = False
model.classifier = nn.Sequential(
    nn.Dropout(0.3),
    nn.Linear(model.last_channel, 128),
    nn.ReLU(),
    nn.Dropout(0.2),
    nn.Linear(128, 2),
)
model = model.to(DEVICE)

dataset    = GarmentDataset(train_tf)
dataloader = DataLoader(dataset, batch_size=16, shuffle=True, num_workers=2)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.classifier.parameters(), lr=1e-3, weight_decay=1e-4)

EPOCHS   = 15
best_acc = 0.0
ckpt_path = f'{BASE}/models/cv_model/classifier.pth'

print(f'\nTraining for {EPOCHS} epochs on {len(dataset)} images ...')
print()
model.train()
for epoch in range(EPOCHS):
    total_loss = correct = total = 0
    for images, labels in dataloader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        out  = model(images)
        loss = criterion(out, labels)
        loss.backward(); optimizer.step()
        total_loss += loss.item()
        correct    += (out.argmax(1) == labels).sum().item()
        total      += labels.size(0)
    acc = round(correct / total * 100, 1)
    tag = ''
    if acc > best_acc:
        best_acc = acc
        torch.save(model.state_dict(), ckpt_path)
        tag = ' best saved'
    print(f'  Epoch {epoch+1:02d}/{EPOCHS} '
          f'- Loss: {total_loss/len(dataloader):.4f} '
          f'- Acc: {acc}%{tag}')

print()
print(f' Best accuracy: {best_acc}%')
print(f'Classifier saved: {ckpt_path}')

# extract visual features
print('\nExtracting visual features ...')
model.eval()
feature_extractor = nn.Sequential(*list(model.children())[:-1]).to(DEVICE)
feature_extractor.eval()

df_prods = pd.read_csv(f'{BASE}/data/raw/metadata/products.csv')
proc_dir = f'{BASE}/data/processed/images'
proc_files = sorted([f for f in os.listdir(proc_dir) if f.endswith('.jpg')])

features = {}
with torch.no_grad():
    for idx, row in df_prods.iterrows():
        pid, gtype = row['product_id'], row['garment_type']
        matching = [f for f in proc_files if f.startswith(gtype)] or proc_files
        fname    = matching[idx % len(matching)]
        try:
            img    = Image.open(os.path.join(proc_dir, fname)).convert('RGB')
            tensor = eval_tf(img).unsqueeze(0).to(DEVICE)
            vec    = feature_extractor(tensor).squeeze().cpu().numpy().flatten()
            features[pid] = vec
        except Exception as e:
            print(f'  ⚠ {pid}: {e}')
        if (idx+1) % 25 == 0:
            print(f'  {idx+1}/100 features extracted ...')

pickle.dump(features, open(f'{BASE}/models/cv_model/visual_features.pkl','wb'))
print(f'Visual features saved vector size: {list(features.values())[0].shape}')
print()


 Device: cuda
 Training images — abaya: 250, thobe: 250

Training for 15 epochs on 500 images ...

  Epoch 01/15 - Loss: 0.4849 - Acc: 75.2% best saved
  Epoch 02/15 - Loss: 0.2543 - Acc: 91.8% best saved
  Epoch 03/15 - Loss: 0.2096 - Acc: 92.2% best saved
  Epoch 04/15 - Loss: 0.2585 - Acc: 90.6%
  Epoch 05/15 - Loss: 0.1695 - Acc: 93.6% best saved
  Epoch 06/15 - Loss: 0.1560 - Acc: 93.4%
  Epoch 07/15 - Loss: 0.1748 - Acc: 93.8% best saved
  Epoch 08/15 - Loss: 0.1864 - Acc: 93.0%
  Epoch 09/15 - Loss: 0.1935 - Acc: 91.6%
  Epoch 10/15 - Loss: 0.1531 - Acc: 93.8%
  Epoch 11/15 - Loss: 0.1936 - Acc: 94.6% best saved
  Epoch 12/15 - Loss: 0.2035 - Acc: 91.6%
  Epoch 13/15 - Loss: 0.1445 - Acc: 93.4%
  Epoch 14/15 - Loss: 0.1266 - Acc: 95.4% best saved
  Epoch 15/15 - Loss: 0.1262 - Acc: 95.4%

 Best accuracy: 95.4%
Classifier saved: /content/drive/MyDrive/thobe_recommender/models/cv_model/classifier.pth

Extracting visual features ...
  25/100 features extracted ...
  50/100 features

## Write FastAPI Server to Disk

In [ ]:
import os

BASE = '/content/drive/MyDrive/thobe_recommender'

api_code = '''
import os, sys, pickle, io
BASE = "/content/drive/MyDrive/thobe_recommender"
sys.path.insert(0, BASE)
os.chdir(BASE)

import numpy as np
import pandas as pd
from fastapi import FastAPI, UploadFile, File, Query, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from typing import Optional
from PIL import Image
import torch, torch.nn as nn
import torchvision.models as models
import torchvision.transforms as transforms

app = FastAPI(title="Thobe & Abaya Recommender", version="1.0.0")
app.add_middleware(CORSMiddleware, allow_origins=["*"],
                   allow_methods=["*"], allow_headers=["*"])

print("Loading models ...")
svd_model = pickle.load(open(f"{BASE}/models/recommender/svd_model.pkl",    "rb"))
cos_sim   = pickle.load(open(f"{BASE}/models/recommender/cosine_sim.pkl",   "rb"))
df_prods  = pickle.load(open(f"{BASE}/models/recommender/products_df.pkl",  "rb"))
vis_feats = pickle.load(open(f"{BASE}/models/cv_model/visual_features.pkl", "rb"))
prod_ids  = df_prods["product_id"].tolist()

U        = svd_model["U"];  V  = svd_model["V"]
bu       = svd_model["bu"]; bi = svd_model["bi"]
mu       = svd_model["mu"]
user2idx = svd_model["user2idx"]
item2idx = svd_model["item2idx"]

DEVICE  = torch.device("cpu")
CLASSES = ["abaya", "thobe"]

classifier = models.mobilenet_v2(weights=None)
classifier.classifier = nn.Sequential(
    nn.Dropout(0.3), nn.Linear(classifier.last_channel, 128),
    nn.ReLU(), nn.Dropout(0.2), nn.Linear(128, 2))
classifier.load_state_dict(torch.load(
    f"{BASE}/models/cv_model/classifier.pth", map_location=DEVICE))
classifier.eval()
feature_extractor = nn.Sequential(*list(classifier.children())[:-1])
feature_extractor.eval()

eval_tf = transforms.Compose([
    transforms.Resize((224,224)), transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])

interactions = {}
print(f"Ready — {len(prod_ids)} products loaded")

def detect_garment_type(img):
    t = eval_tf(img).unsqueeze(0)
    with torch.no_grad():
        out   = classifier(t)
        probs = torch.softmax(out, dim=1)[0]
    return CLASSES[out.argmax(1).item()], round(probs.max().item()*100,1)

def extract_features(img):
    t = eval_tf(img).unsqueeze(0)
    with torch.no_grad():
        return feature_extractor(t).squeeze().numpy().flatten()

def get_metadata(pid):
    row = df_prods[df_prods["product_id"]==pid]
    if row.empty: return {}
    r = row.iloc[0]
    return {k: (float(r[k]) if k=="price" else r[k])
            for k in ["garment_type","occasion","fabric","color","pattern",
                      "sleeve_length","embellishment","price_range",
                      "title","brand","price","currency"]}

def svd_score(uid, pid):
    u = user2idx.get(uid); i = item2idx.get(pid)
    if u is None or i is None: return 0.5
    return float(np.clip(mu+bu[u]+bi[i]+U[u]@V[i], 1, 5)) / 5.0

def hybrid_score(uid, pid, cidx, a=0.4, b=0.4, g=0.2):
    trend = sum(1 for v in interactions.values()
                if v.get(pid,{}).get("event")=="purchase")
    return round(a*svd_score(uid,pid)
                +b*float(np.mean(cos_sim[cidx]))
                +g*min(trend/10.0,1.0), 4)

@app.get("/v1/health")
def health():
    return {"status":"ok","total_products":len(prod_ids),"version":"1.0.0"}

class RecoRequest(BaseModel):
    user_id: str; n: int = 5; garment_type: Optional[str] = None

@app.post("/v1/recommend")
def recommend(req: RecoRequest):
    results = []
    for i,pid in enumerate(prod_ids):
        meta = get_metadata(pid)
        if req.garment_type and meta.get("garment_type")!=req.garment_type: continue
        results.append({"product_id":pid,"score":hybrid_score(req.user_id,pid,i),"metadata":meta})
    results.sort(key=lambda x:x["score"], reverse=True)
    return {"user_id":req.user_id,"recommendations":results[:req.n],"version":"1.0.0"}

@app.post("/v1/recommend/image")
async def recommend_image(file: UploadFile=File(...),
                          user_id: str=Query(...), n: int=Query(5)):
    img          = Image.open(io.BytesIO(await file.read())).convert("RGB")
    gtype, conf  = detect_garment_type(img)
    q_vec        = extract_features(img)
    results = []
    for pid, fvec in vis_feats.items():
        if get_metadata(pid).get("garment_type") != gtype: continue
        ml = min(len(q_vec), len(fvec))
        v1,v2 = q_vec[:ml], fvec[:ml]
        sim = float(np.dot(v1,v2)/(np.linalg.norm(v1)*np.linalg.norm(v2)+1e-9))
        results.append({"product_id":pid,"score":round(sim,4),"metadata":get_metadata(pid)})
    results.sort(key=lambda x:x["score"], reverse=True)
    return {"user_id":user_id,"detected_type":gtype,
            "confidence":f"{conf}%","recommendations":results[:n],"version":"1.0.0"}

class InteractionRequest(BaseModel):
    user_id: str; product_id: str; interaction_type: str

@app.post("/v1/interaction")
def log_interaction(req: InteractionRequest):
    interactions.setdefault(req.user_id,{})[req.product_id] = {"event":req.interaction_type}
    return {"status":"logged","user_id":req.user_id,"product_id":req.product_id}

@app.get("/v1/trending")
def trending(n: int=Query(5), garment_type: Optional[str]=Query(None)):
    counts = {}
    for u in interactions.values():
        for pid,evt in u.items():
            if evt.get("event")=="purchase": counts[pid]=counts.get(pid,0)+1
    results = []
    for pid,cnt in sorted(counts.items(),key=lambda x:x[1],reverse=True):
        meta = get_metadata(pid)
        if garment_type and meta.get("garment_type")!=garment_type: continue
        results.append({"product_id":pid,"trend_score":round(cnt/10,4),"metadata":meta})
        if len(results)>=n: break
    return {"trending":results}

@app.get("/v1/similar/{product_id}")
def similar(product_id: str, n: int=Query(5)):
    if product_id not in prod_ids:
        raise HTTPException(404, f"{product_id} not found")
    idx  = prod_ids.index(product_id)
    sims = sorted(enumerate(cos_sim[idx]),key=lambda x:x[1],reverse=True)
    return {"product_id":product_id,
            "similar":[{"product_id":prod_ids[i],"score":round(s,4),
                        "metadata":get_metadata(prod_ids[i])}
                       for i,s in sims[1:n+1]]}
'''

os.makedirs(f'{BASE}/src/api', exist_ok=True)
with open(f'{BASE}/src/api/main.py', 'w') as f:
    f.write(api_code)

print('src/api/main.py written')
print()


src/api/main.py written



## Launch Server & Get Public URL

In [ ]:
import subprocess, threading, time, os, requests, re

BASE = '/content/drive/MyDrive/thobe_recommender'
os.chdir(BASE)

# kill any leftover server on port 8000
os.system('fuser -k 8000/tcp 2>/dev/null || true')
time.sleep(1)

# start uvicorn in background
log_path = f'{BASE}/api.log'
api_proc = subprocess.Popen(
    ['python', '-m', 'uvicorn', 'src.api.main:app',
     '--host', '0.0.0.0', '--port', '8000'],
    stdout=open(log_path, 'w'),
    stderr=subprocess.STDOUT,
    cwd=BASE
)
print('Waiting for server to start ...')
ready = False
for i in range(20):
    time.sleep(3)
    try:
        r = requests.get('http://localhost:8000/v1/health', timeout=2)
        if r.status_code == 200:
            print('✓ Server is up!'); ready = True; break
    except:
        pass
    print(f'  {(i+1)*3}s ...')

if not ready:
    print('\n⚠ Server failed to start. Logs:')
    with open(log_path) as f: print(f.read())
else:
    os.system('wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /tmp/cloudflared')
    os.system('chmod +x /tmp/cloudflared')

    tunnel_url = {'url': None}

    def run_tunnel():
        p = subprocess.Popen(
            ['/tmp/cloudflared', 'tunnel', '--url', 'http://localhost:8000'],
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            universal_newlines=True
        )
        for line in p.stdout:
            m = re.search(r'https://[a-z0-9\-]+\.trycloudflare\.com', line)
            if m:
                tunnel_url['url'] = m.group(0)

    threading.Thread(target=run_tunnel, daemon=True).start()

    print('Waiting for tunnel ...')
    for i in range(20):
        if tunnel_url['url']: break
        time.sleep(2)
        print(f'  {(i+1)*2}s ...')

    url = tunnel_url['url']
    if url:
        with open(f'{BASE}/tunnel_url.txt', 'w') as f:
            f.write(url)
        print()
        print(f' API IS LIVE!')
        print()
        print(f'  Swagger UI - {url}/docs')
        print(f'  Health - {url}/v1/health')
        print()
    else:
        print('Tunnel failed — check logs above')

Waiting for server to start ...
  3s ...
✓ Server is up!
Waiting for tunnel ...
  2s ...
  4s ...
  6s ...
  8s ...
  10s ...

 API IS LIVE!

  Swagger UI - https://inclusion-adventures-thrown-plumbing.trycloudflare.com/docs
  Health - https://inclusion-adventures-thrown-plumbing.trycloudflare.com/v1/health



## Test All API Endpoints

In [ ]:
import requests, json, io
from PIL import Image, ImageDraw

BASE = '/content/drive/MyDrive/thobe_recommender'
with open(f'{BASE}/tunnel_url.txt') as f:
    url = f.read().strip()
print(f'Testing: {url}\n')

# Health
print(); print('1. HEALTH CHECK'); print()
print(json.dumps(requests.get(f'{url}/v1/health').json(), indent=2))

# 2. Hybrid recommend abayas
print(); print('2. ABAYA RECS for user_0001'); print()
r = requests.post(f'{url}/v1/recommend',
                  json={'user_id':'user_0001','garment_type':'abaya','n':5})
for i,rec in enumerate(r.json()['recommendations'],1):
    m = rec['metadata']
    print(f"{i}. {rec['product_id']} | {m['color']} {m['pattern']} {m['garment_type']}"
          f" | {m['fabric']} | {m['price']} SAR | score: {rec['score']}")

# 3. Hybrid recommend thobes
print(); print('3. THOBE RECS for user_0050'); print()
r = requests.post(f'{url}/v1/recommend',
                  json={'user_id':'user_0050','garment_type':'thobe','n':5})
for i,rec in enumerate(r.json()['recommendations'],1):
    m = rec['metadata']
    print(f"{i}. {rec['product_id']} | {m['color']} {m['pattern']} {m['garment_type']}"
          f" | {m['fabric']} | {m['price']} SAR | score: {rec['score']}")

# 4. Image-based recommendation
print(); print('4. IMAGE-BASED RECOMMENDATION'); print()
img = Image.new('RGB', (224,224), (15,15,15))
ImageDraw.Draw(img).polygon([(112,20),(60,80),(40,200),(184,200),(164,80)], outline='white')
buf = io.BytesIO(); img.save(buf,'JPEG'); buf.seek(0)
r = requests.post(f'{url}/v1/recommend/image',
                  params={'user_id':'user_0002','n':3},
                  files={'file':('test.jpg',buf,'image/jpeg')})
data = r.json()
print(f"Auto-detected: {data.get('detected_type')} ({data.get('confidence')})")
for i,rec in enumerate(data['recommendations'],1):
    m = rec['metadata']
    print(f"{i}. {rec['product_id']} | {m['color']} {m['garment_type']} | score: {rec['score']}")

# 5. Log interaction
print(); print('5. LOG INTERACTION'); print()
r = requests.post(f'{url}/v1/interaction',
                  json={'user_id':'user_0001','product_id':'p000','interaction_type':'purchase'})
print(json.dumps(r.json(), indent=2))

# 6. Trending
print(); print('6. TRENDING PRODUCTS'); print()
r = requests.get(f'{url}/v1/trending?n=3')
data = r.json()
if data['trending']:
    for item in data['trending']:
        m = item['metadata']
        print(f"- {item['product_id']} | {m['garment_type']} | trend: {item['trend_score']}")
else:
    print('No trending yet (need purchase interactions first)')

# 7. Similar items
print(); print('7. SIMILAR TO p000'); print()
r = requests.get(f'{url}/v1/similar/p000?n=3')
for item in r.json()['similar']:
    m = item['metadata']
    print(f"- {item['product_id']} | {m['color']} {m['garment_type']} | score: {item['score']}")

print()
print('ALL ENDPOINTS WORKING!')
print()
print(f'  Swagger UI  → {url}/docs')
print()


Testing: https://inclusion-adventures-thrown-plumbing.trycloudflare.com


1. HEALTH CHECK

{
  "status": "ok",
  "total_products": 100,
  "version": "1.0.0"
}

2. ABAYA RECS for user_0001

1. p030 | grey printed abaya | chiffon | 113.46 SAR | score: 0.2925
2. p014 | grey plain abaya | linen | 166.61 SAR | score: 0.2892
3. p062 | black plain abaya | cotton | 108.59 SAR | score: 0.2885
4. p028 | navy printed abaya | linen | 105.09 SAR | score: 0.288
5. p056 | grey geometric abaya | silk | 177.34 SAR | score: 0.2828

3. THOBE RECS for user_0050

1. p071 | cream printed thobe | crepe | 527.27 SAR | score: 0.2997
2. p015 | white striped thobe | polyester | 668.73 SAR | score: 0.2925
3. p003 | cream striped thobe | chiffon | 142.97 SAR | score: 0.2915
4. p091 | grey embroidered thobe | cotton | 141.68 SAR | score: 0.2911
5. p041 | cream printed thobe | cotton | 115.58 SAR | score: 0.2908

4. IMAGE-BASED RECOMMENDATION

Auto-detected: abaya (81.2%)
1. p010 | black abaya | score: 0.3424
2. p06